# Groundhog Day Predictions - Exploratory Data Analysis

This notebook explores the Groundhog Day API data for Bayesian analysis.

**Data Source:** https://groundhog-day.com/api/v1/

**Key Variables:**
- `shadow`: 1 = saw shadow (6 more weeks of winter), 0 = no shadow (early spring)
- `year`: Year of prediction (1886-2022)
- `groundhog`: Metadata about each prognosticating animal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Libraries imported successfully")

## Load and Process Data

In [ ]:
# Load the combined data
with open('data/combined_data.json', 'r') as f:
    data = json.load(f)

# Extract groundhogs info
groundhogs = data['groundhogs']
predictions_by_year = data['predictions_by_year']

print(f"Number of groundhogs: {len(groundhogs)}")
print(f"Year range: {data['metadata']['year_range']}")
print(f"Fetched at: {data['metadata']['fetched_at']}")

In [ ]:
# Create a flat DataFrame of all predictions
all_predictions = []

for year, year_payload in predictions_by_year.items():
    year_preds = year_payload.get('predictions') if isinstance(year_payload, dict) else year_payload
    if not year_preds:
        continue
    for pred in year_preds:
        pred = dict(pred)
        pred['year'] = int(year)
        all_predictions.append(pred)

df = pd.DataFrame(all_predictions)

# Extract groundhog info into separate columns
groundhog_df = pd.json_normalize(df['groundhog'])
df = pd.concat([df.drop('groundhog', axis=1), groundhog_df], axis=1)

print(f"Total predictions: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
df

In [ ]:
# Display basic info about the dataset
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
df.head(10)

In [ ]:
# Clean up the shadow column (handle nulls)
print("Shadow value counts:")
print(df['shadow'].value_counts(dropna=False))

# Create binary shadow indicator (excluding nulls)
df['shadow_binary'] = df['shadow'].apply(lambda x: 1 if x == 1 else (0 if x == 0 else np.nan))
print(f"\nValid predictions (non-null shadow): {df['shadow_binary'].notna().sum()}")

## Overall Shadow Statistics

In [ ]:
# Calculate overall shadow rate
valid_df = df[df['shadow_binary'].notna()]
overall_shadow_rate = valid_df['shadow_binary'].mean()

print("=" * 50)
print("OVERALL SHADOW STATISTICS")
print("=" * 50)
print(f"Total valid predictions: {len(valid_df)}")
print(f"Saw shadow: {valid_df['shadow_binary'].sum():.0f} ({overall_shadow_rate*100:.1f}%)")
print(f"No shadow: {(1-valid_df['shadow_binary']).sum():.0f} ({(1-overall_shadow_rate)*100:.1f}%)")

In [ ]:
# Year-by-year shadow rate
yearly_stats = valid_df.groupby('year').agg({
    'shadow_binary': ['sum', 'count', 'mean']
}).reset_index()
yearly_stats.columns = ['year', 'shadow_count', 'total_count', 'shadow_rate']

print("\nYearly statistics (first 20 years):")
print(yearly_stats.head(20))

## Visualizations

In [ ]:
# Time series of shadow predictions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Shadow rate over time (rolling average)
ax1 = axes[0, 0]
yearly_stats['rolling_rate'] = yearly_stats['shadow_rate'].rolling(window=10, min_periods=1).mean()
ax1.plot(yearly_stats['year'], yearly_stats['shadow_rate'], 'o-', alpha=0.5, label='Yearly Rate')
ax1.plot(yearly_stats['year'], yearly_stats['rolling_rate'], '-', linewidth=2, label='10-Year Rolling Avg')
ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='50% Baseline')
ax1.set_xlabel('Year')
ax1.set_ylabel('Shadow Rate')
ax1.set_title('Groundhog Shadow Prediction Rate Over Time')
ax1.legend()
ax1.set_ylim(0, 1)

# Plot 2: Distribution of yearly shadow rates
ax2 = axes[0, 1]
ax2.hist(yearly_stats['shadow_rate'], bins=20, edgecolor='black', alpha=0.7)
ax2.axvline(x=overall_shadow_rate, color='red', linestyle='--', linewidth=2, label=f'Overall Mean: {overall_shadow_rate:.3f}')
ax2.axvline(x=0.5, color='green', linestyle='--', linewidth=2, label='50% Baseline')
ax2.set_xlabel('Shadow Rate')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Yearly Shadow Rates')
ax2.legend()

# Plot 3: Cumulative shadow predictions
ax3 = axes[1, 0]
valid_df_sorted = valid_df.sort_values('year')
cumulative_shadows = valid_df_sorted['shadow_binary'].cumsum()
ax3.plot(valid_df_sorted['year'], cumulative_shadows, '-')
ax3.plot(valid_df_sorted['year'], np.arange(1, len(cumulative_shadows)+1)*0.5, '--', color='green', label='50% Expected')
ax3.set_xlabel('Year')
ax3.set_ylabel('Cumulative Shadow Predictions')
ax3.set_title('Cumulative Shadow Predictions Over Time')
ax3.legend()

# Plot 4: Bar chart of decades
ax4 = axes[1, 1]
valid_df['decade'] = (valid_df['year'] // 10) * 10
decade_stats = valid_df.groupby('decade')['shadow_binary'].mean()
decade_stats.plot(kind='bar', ax=ax4, edgecolor='black')
ax4.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='50% Baseline')
ax4.set_xlabel('Decade')
ax4.set_ylabel('Shadow Rate')
ax4.set_title('Shadow Rate by Decade')
ax4.legend()
ax4.set_ylim(0, 1)
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('data/shadow_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to data/shadow_analysis.png")

## Groundhog Analysis

In [ ]:
# Analyze groundhog types and their predictions
print("=" * 50)
print("GROUNDHOG ANALYSIS")
print("=" * 50)

# Groundhog types
print("\nGroundhog Types:")
print(df['type'].value_counts())

# Country distribution
print("\nCountry Distribution:")
print(df['country'].value_counts())

In [ ]:
# Analyze predictions by groundhog
groundhog_stats = valid_df.groupby(['slug', 'shortname', 'type', 'country']).agg({
    'shadow_binary': ['count', 'sum', 'mean']
}).reset_index()
groundhog_stats.columns = ['slug', 'shortname', 'type', 'country', 'predictions', 'shadows', 'shadow_rate']
groundhog_stats = groundhog_stats.sort_values('predictions', ascending=False)

print("\nTop 15 Most Active Groundhogs:")
print(groundhog_stats.head(15).to_string())

In [ ]:
# Shadow rate by groundhog type
type_stats = valid_df.groupby('type')['shadow_binary'].agg(['count', 'mean', 'std']).reset_index()
type_stats.columns = ['type', 'count', 'mean_shadow_rate', 'std']
type_stats = type_stats.sort_values('count', ascending=False)

print("\nShadow Rate by Groundhog Type:")
print(type_stats.to_string())

# Create visualization
fig, ax = plt.subplots(figsize=(12, 6))
type_stats_sorted = type_stats.sort_values('mean_shadow_rate')
colors = plt.cm.RdYlGn(type_stats_sorted['mean_shadow_rate'])
bars = ax.barh(type_stats_sorted['type'], type_stats_sorted['mean_shadow_rate'], color=colors, edgecolor='black')
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='50% Baseline')
ax.set_xlabel('Shadow Rate')
ax.set_ylabel('Groundhog Type')
ax.set_title('Shadow Rate by Groundhog Type')
ax.legend()
ax.set_xlim(0, 1)

# Add count labels
for i, (rate, count) in enumerate(zip(type_stats_sorted['mean_shadow_rate'], type_stats_sorted['count'])):
    ax.text(rate + 0.02, i, f'n={count}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('data/groundhog_type_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Statistical Tests

In [ ]:
# One-sample binomial test: Is shadow rate significantly different from 50%?
from scipy.stats import binomtest, chi2_contingency

n_shadows = valid_df['shadow_binary'].sum()
n_total = len(valid_df)

# Binomial test
binom_result = binomtest(int(n_shadows), n_total, p=0.5, alternative='two-sided')

print("=" * 50)
print("STATISTICAL TESTS")
print("=" * 50)

print("\n1. BINOMIAL TEST (Is shadow rate ≠ 50%?)")
print(f"   H0: Shadow rate = 0.5")
print(f"   H1: Shadow rate ≠ 0.5")
print(f"   Number of shadows: {n_shadows}")
print(f"   Total predictions: {n_total}")
print(f"   Observed proportion: {n_shadows/n_total:.4f}")
print(f"   p-value: {binom_result.pvalue:.6f}")
if binom_result.pvalue < 0.05:
    print(f"   Result: REJECT H0 (significant at α=0.05)")
else:
    print(f"   Result: FAIL TO REJECT H0")

In [ ]:
# Chi-square test for trend over time
from scipy.stats import chi2_contingency

# Divide years into periods
valid_df['period'] = pd.cut(valid_df['year'], 
                           bins=[1885, 1950, 1980, 2000, 2025],
                           labels=['1886-1950', '1951-1980', '1981-2000', '2001-2022'])

# Create contingency table
contingency = pd.crosstab(valid_df['period'], valid_df['shadow_binary'])
print("\n2. CHI-SQUARE TEST (Is shadow rate changing over time?)")
print("   Contingency Table:")
print(contingency)

chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f"\n   Chi-square statistic: {chi2:.4f}")
print(f"   p-value: {p_value:.6f}")
print(f"   Degrees of freedom: {dof}")
if p_value < 0.05:
    print(f"   Result: REJECT H0 (significant association)")
else:
    print(f"   Result: FAIL TO REJECT H0")

## Bayesian Analysis Prep

In [ ]:
# Prepare data for Bayesian analysis
# Create a summary dataset

bayesian_data = {
    'overall': {
        'n_shadows': int(n_shadows),
        'n_total': n_total,
        'proportion': n_shadows/n_total
    },
    'yearly_data': yearly_stats[['year', 'shadow_rate', 'shadow_count', 'total_count']].to_dict('records'),
    'decade_data': valid_df.groupby('decade')['shadow_binary'].agg(['sum', 'count', 'mean']).reset_index().to_dict('records'),
    'groundhog_data': groundhog_stats.to_dict('records'),
    'period_data': valid_df.groupby('period')['shadow_binary'].agg(['sum', 'count', 'mean']).reset_index().to_dict('records')
}

# Save for later Bayesian analysis
with open('data/bayesian_input.json', 'w') as f:
    json.dump(bayesian_data, f, indent=2)

print("Bayesian analysis input saved to data/bayesian_input.json")

print("\n" + "=" * 50)
print("SUMMARY STATISTICS FOR BAYESIAN ANALYSIS")
print("=" * 50)
print(f"\nOverall shadow rate: {n_shadows/n_total:.4f}")
print(f"95% CI (normal approximation): {n_shadows/n_total - 1.96*np.sqrt((n_shadows/n_total)*(1-n_shadows/n_total)/n_total):.4f} to {n_shadows/n_total + 1.96*np.sqrt((n_shadows/n_total)*(1-n_shadows/n_total)/n_total):.4f}")

In [ ]:
# Summary statistics
print("\nYearly Shadow Rate Statistics:")
print(f"  Mean: {yearly_stats['shadow_rate'].mean():.4f}")
print(f"  Std Dev: {yearly_stats['shadow_rate'].std():.4f}")
print(f"  Min: {yearly_stats['shadow_rate'].min():.4f}")
print(f"  Max: {yearly_stats['shadow_rate'].max():.4f}")

# Decade-wise summary
print("\nDecade-wise Shadow Rates:")
decade_summary = valid_df.groupby('decade')['shadow_binary'].agg(['sum', 'count', 'mean'])
decade_summary.columns = ['Shadows', 'Total', 'Rate']
print(decade_summary)

## Next Steps for Bayesian Analysis

The data is now ready for Bayesian analysis. Possible analyses include:

1. **Beta-Binomial Model**: Estimate overall shadow probability with uncertainty
2. **Hierarchical Model**: Model shadow rates by decade or groundhog type
3. **Time Series Model**: Account for temporal autocorrelation
4. **Bayesian Hypothesis Testing**: Compare against null hypothesis of 50%

You can use PyMC3, PyMC, or Stan for the Bayesian modeling.